# Lab 0-04: Tools Give Agents Specific Capabilities

An `LLM` receives text and generates text. To retrieve current information, perform a calculation reliably, create media, access a file, or interact with another service, an agent needs a **tool**: a function that provides one clear capability.

In this notebook, you will explore a small multiplication tool and a weather tool with fixed demonstration data. No model, API key, or internet connection is needed.

This notebook adapts the core ideas in the Hugging Face Agents Course lesson, ["What are Tools?"](https://huggingface.co/learn/agents-course/en/unit1/tools).

## What You Will Learn

By the end, you should be able to:

- explain why a tool is different from an `LLM` response
- identify a tool's name, purpose, inputs, outputs, and limits
- follow the basic tool-use loop: the model requests a tool, the agent runs it, and the result becomes new context
- explain why a real weather tool needs an approved source of current information

## 1. What Are AI Tools?

A tool is a function made available to an `LLM` through an agent. It should have one clear objective. The agent can provide many different tools, depending on the task.

| Tool | What it lets an agent do |
| --- | --- |
| Web search | Retrieve current information from the internet. |
| Image generation | Create an image from a text description. |
| Retrieval | Find information in an external source, such as an approved document collection. |
| API interface | Interact with another service, such as GitHub, YouTube, or Spotify. |

These are only examples. A developer can create a tool for any well-defined use case.

A good tool complements the capabilities of an `LLM`. Two common reasons to use one are:

- **Reliable computation:** A calculator tool can produce arithmetic results more reliably than asking the model to calculate in its text response.
- **Current information:** When an answer depends on information that may have changed, an agent can use an approved tool to retrieve it from an appropriate external source, such as a weather provider. The `LLM` then uses the returned result as context rather than guessing.

![Weather tool request and response](figures/weather_tool.png)

*Figure 1. A weather tool receives a location, obtains a weather report, and returns the result for the agent to use as context.*

### 1.1 A Multiplication Tool Example

This small calculator tool illustrates the reliable-computation use case. It has a narrow objective: multiply two whole numbers and return the result.

In [ ]:
def multiply_integers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

multiply_integers(12, 7)

### 1.2 A Weather Tool Example

The following function demonstrates the shape of a weather tool. As Figure 1 shows, in a real agent the tool would send a location to an approved weather provider's API, receive current weather data, and return that result to the agent.

To keep this notebook reliable in every classroom, this version uses fixed demonstration data instead of contacting a live weather service.

In [ ]:
DEMO_WEATHER = {
    "Paris": {"condition": "partly cloudy", "temperature_c": 18},
    "New York": {"condition": "light rain", "temperature_c": 16},
}


def weather_tool(location: str) -> dict[str, str | int]:
    """Return a simulated weather report for a named location."""
    if location not in DEMO_WEATHER:
        raise ValueError(f"No demonstration weather data for {location}.")
    return {"location": location, **DEMO_WEATHER[location]}

weather_tool("Paris")

## 2. How Do Tools Work?

**The key issue:** a tool is ordinary code, such as a Python function, but an `LLM` receives text and generates text. The `LLM` cannot directly call a Python function or contact a provider on its own.

The agent program bridges this gap. It describes the available tools to the `LLM`, receives a text-based tool request, checks it, and then runs the approved Python function.

For a user who asks for the product of 12 and 7, the tool-use loop is:

1. **The `LLM` proposes a tool request.** It recognizes that the multiplication tool would help and produces text representing `multiply_integers(12, 7)`.
2. **The agent validates the request.** It checks that the multiplication tool is available and approved.
3. **The agent executes the tool.** It runs the function on the model's behalf and receives the result, `84`.
4. **The result becomes context.** The agent adds `84` to the conversation and asks the `LLM` to write a natural-language response using that result.

Most applications keep these intermediate tool steps out of the user interface. From the user's perspective, the model appears to have calculated the answer itself, but the agent program actually performed the tool call in the background.

In compact form: `user question` → `LLM tool request` → `agent validates and runs tool` → `tool result becomes context` → `LLM response`.

**Important:** The `LLM`'s tool request must follow the agent's tool instructions and schema. It needs an approved tool name and arguments that match the function parameters. Otherwise, the request can fail, so the agent validates it before running any code.

In [ ]:
# This dictionary represents a constrained tool request that an LLM could have generated.
# Its required fields and allowed values come from the agent's tool instructions.
# The tool name tells the agent which approved function the model wants to use.
tool_request = {
    "name": "multiply_integers",
    # These argument names must match the function parameters: a and b.
    "arguments": {"a": 12, "b": 7},
}

# The agent keeps an allowlist of the tools this workflow permits.
approved_tool_names = {"multiply_integers"}
# Stop if the model requested a tool that is not on the allowlist.
if tool_request["name"] not in approved_tool_names:
    raise ValueError("Requested tool is not approved for this workflow.")

# ** unpacks {"a": 12, "b": 7} into multiply_integers(a=12, b=7).
tool_result = multiply_integers(**tool_request["arguments"])
# In a full agent, this result would be sent back to the LLM as new context.
print("Tool returned:\n")
print(tool_result)

## 3. How Do We Give Tools to an LLM?

The surrounding agent usually includes descriptions of the available tools in the model's persistent instructions. The description must state what the tool does and exactly which inputs it expects. Structured formats such as JSON or a programming-language signature make those details less ambiguous.

The next cell creates a simple description that an agent could provide to an `LLM`.

In [ ]:
tool_specification = {
    "name": "multiply_integers",
    "description": "Multiply two integers.",
    "inputs": {"a": "first integer", "b": "second integer"},
    "output": "one integer product",
    "limits": [
        "Multiplies only two integer inputs",
        "Does not retrieve external information",
    ],
}

for label, value in tool_specification.items():
    print(f"{label}: {value}")

### 3.1 Auto-formatting Tool Sections

Writing every tool description by hand is repetitive and easy to get wrong. A well-written Python function already contains much of the needed information: a meaningful name, a docstring, typed inputs, and an output type. The next helper inspects those details and turns them into a consistent description.

This is the idea behind many tool libraries: developers write ordinary Python functions, and the library builds a consistent tool description for the `LLM`.

In [ ]:
import inspect

def type_name(annotation) -> str:
    """Return a readable name for a Python type annotation."""
    return getattr(annotation, "__name__", str(annotation))


def format_tool_description(function) -> str:
    """Build an LLM-readable tool description from a Python function."""
    signature = inspect.signature(function)
    arguments = [
        f"{parameter.name}: {type_name(parameter.annotation)}"
        for parameter in signature.parameters.values()
    ]
    description = inspect.getdoc(function) or "No description provided."
    output = type_name(signature.return_annotation)
    return (
        f"Tool Name: {function.__name__}\n"
        f"Description: {description}\n"
        f"Arguments: {', '.join(arguments)}\n"
        f"Output: {output}"
    )

print(format_tool_description(multiply_integers))

### 3.2 Generic Tool Implementation

A generic `Tool` class gives each tool the same interface: a name, a description, inputs, an output, and a callable function. The `@tool` decorator below converts an ordinary typed, documented Python function into a `Tool` object automatically. You do not need to memorize the implementation. Focus on the result: a clear function definition becomes a reusable, consistently described capability.

In [ ]:
from collections.abc import Callable


class Tool:
    """A reusable wrapper around one approved Python function."""

    def __init__(self, name: str, description: str, function: Callable, arguments: list[str], output: str):
        self.name = name
        self.description = description
        self.function = function
        self.arguments = arguments
        self.output = output

    def to_string(self) -> str:
        """Return the information an LLM needs to know about this tool."""
        return (
            f"Tool Name: {self.name}\n"
            f"Description: {self.description}\n"
            f"Arguments: {', '.join(self.arguments)}\n"
            f"Output: {self.output}"
        )

    def __call__(self, *arguments, **keyword_arguments):
        """Run the wrapped function after the agent has approved the call."""
        return self.function(*arguments, **keyword_arguments)


def tool(function: Callable) -> Tool:
    """Turn a typed, documented function into a reusable Tool object."""
    signature = inspect.signature(function)
    arguments = [
        f"{parameter.name}: {type_name(parameter.annotation)}"
        for parameter in signature.parameters.values()
    ]
    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "No description provided.",
        function=function,
        arguments=arguments,
        output=type_name(signature.return_annotation),
    )


@tool
def multiply_integers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(multiply_integers.to_string())
print(multiply_integers(12, 7))

### 3.3 Model Context Protocol (MCP): A Unified Tool Interface

An individual tool can work inside one application. **Model Context Protocol (MCP)** is an open protocol that standardizes how applications present tools and other context to `LLMs`. You do not need to use MCP in this lab. The big idea is that a standard interface can let different models and agent frameworks use the same approved tools without each application inventing a separate connection method.

For forensic work, a standard interface does not remove the need for boundaries: the tool's permissions, approved data scope, logging, and human-review requirements still matter.

## What To Notice

The `LLM` would not receive permission to read every file or make every decision. It receives a precise description of one approved capability. The agent program—not the `LLM`—checks the requested tool and executes it.

A tool can do only what its implementation and description permit. This multiplication tool accepts only two integer inputs and returns their product; it does not retrieve information or take an external action.

**Try it.** Change `12` or `7` in the tool-request cell. Then explain: What inputs did the tool accept? What was it not able to do?

Next, open [04_agent_walkthrough.ipynb](04_agent_walkthrough.ipynb). You will compare a plain model with a bounded agent that uses approved case materials.